In [ ]:
!pip install datasets langchain_community fuzzywuzzy sql_metadata -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
import sqlite3
import sqlparse
import re
from datasets import load_dataset, Dataset
from langchain_community.utilities.sql_database import SQLDatabase
from sql_metadata import Parser
from fuzzywuzzy import process

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
def format_dict(dicionario):
    sep = "', '"
    return '{\n  '+",\n  ".join([f"'{chave}': ['{sep.join(valor)}']" for chave, valor in dicionario.items()])+'\n}'

In [ ]:
from fuzzywuzzy import process
def find_closest_match(reference, all):
    closest_match = process.extractOne(reference, all)
    if closest_match:
        return closest_match[0]
    else:
        return None

In [ ]:
def get_schema_dict(db_path):

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    schema_str = "{\n"

    # obter uma lista de todas as tabelas
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name=table[0]

        # obter informações das colunas da tabela
        cursor.execute(f"PRAGMA table_info('{table_name}')")
        columns = cursor.fetchall()

        schema_str += f"  '{table_name}': ["

        for column in columns:
            column_name = column[1].replace('"','')
            schema_str += f"'{column_name}', "

        schema_str = schema_str.rstrip(", ") # remover a última vírgula

        schema_str += "],\n"

    schema_str = schema_str.rstrip(",\n")
    schema_str += '\n}'

    conn.close()
    return schema_str

In [ ]:
def get_tables_and_columns(db_path, query, schema_dict, pk=False):

    connection_string="sqlite:///"+db_path
    db = SQLDatabase.from_uri(connection_string, sample_rows_in_table_info=0)

    all_tables = [table.lower() for table in db.get_table_names()]
    tables_in_query = [table.lower() for table in Parser(query).tables]

    #conferir se todas as tabelas fazem parte do schema:
    correct_tables = []
    for table in tables_in_query:
      if table not in all_tables:
        closest_match = find_closest_match(table, all_tables)
        if closest_match:
            if closest_match in all_tables:
              correct_tables.append(closest_match)
      else:
        correct_tables.append(table)

    query = query.replace('(',' ').replace(')',' ').replace(',',' ').replace(';',' ').replace('t1.','').replace('t2.',' ').replace('t3.',' ').replace('t4.',' ')
    query = query.lower().split()
    # print(query)
    tokens = []
    for token in query:
      if '.' in token:
        token = token.split('.')[1] #pegar nome da coluna
      tokens.append(token)
    # print(tokens)
    query = tokens

    info = {}

    for table_name in correct_tables:
      if table_name in query:
        info[table_name] = []

        columns = eval(db.run((f'PRAGMA table_info("{table_name}")')))

        for column in columns:
          column_name = column[1].lower()

          if pk and column[5] == 1:  # se for a incluir chave primária, e a coluna for uma chave primária
            info[table_name].append(column_name) #incluir, mesmo se ela não estiver na query

          else:
            if column_name in query: # or table_name+'.'+column_name in query:
               info[table_name].append(column_name)

    ### correct columns names:
    info_correct = {}

    dict_info = eval(format_dict(info))
    tables_min = [table for table in dict_info.keys()]
    schema_correct = eval(schema_dict)
    all_tables = [table for table in schema_correct.keys()]

    for table in tables_min:
      correct_table = find_closest_match(table,all_tables)
      if correct_table:
        correct_table = correct_table
      else:
        correct_table = table

      correct_columns = []

      for column in dict_info[table]:
        correct_column = find_closest_match(column,schema_correct[correct_table])
        if correct_column:
          correct_columns.append(correct_column)
        else:
          correct_columns.append(column)

      info_correct[correct_table] = correct_columns

    return info, info_correct


In [ ]:
def get_SQLDatabase(db_path, num_examples = 0):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    schema_str = ""

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table in tables:

        table_name=table[0]
        cursor.execute(f"PRAGMA table_info('{table_name}')")
        included_columns = cursor.fetchall()

        schema_str += f'CREATE TABLE {table_name.lower()} (\n'

        primary_keys = []
        for column in included_columns:
            column_name = column[1].replace('"','')
            column_type = column[2]
            schema_str += f'        {column_name.lower()} {column_type.upper()},\n'

            if column[5] == 1:
                primary_keys.append(column[1].replace('"',''))

        schema_str = schema_str.rstrip(",\n")

        # Adicionar chaves primárias ao esquema
        if primary_keys:
            primary_keys_str = [pk.replace('"','').lower() for pk in primary_keys]
            primary_keys_str = ", ".join(primary_keys_str)
            schema_str += f',\n        PRIMARY KEY ({primary_keys_str})'


        cursor.execute(f"PRAGMA foreign_key_list('{table_name}')")
        foreign_keys_info = cursor.fetchall()
        for fk in foreign_keys_info:
          try:
              fk_col = fk[3].replace('"','')
              ref_table = fk[2].replace('"','')
              ref_col = fk[4].replace('"','')
              schema_str += f',\n        FOREIGN KEY ({fk_col.lower()}) REFERENCES {ref_table.lower()}({ref_col.lower()})'
          except:
            print(fk)

        schema_str += "\n);\n\n"

        if num_examples > 0:
          cursor.execute(f"SELECT {', '.join([col[1] for col in included_columns])} FROM {table_name} LIMIT {num_examples};")
          rows = cursor.fetchall()
          schema_str += f"/*\n{len(rows)} rows from {table_name} table:\n"
          schema_str += "\t".join([col[1].lower().replace('"','') for col in included_columns]) + "\n"
          for row in rows:
              schema_str += "\t".join(map(str, row)) + "\n"
          schema_str += "*/\n\n"

    schema_str = schema_str.rstrip('\n\n')

    conn.close()
    return schema_str

In [ ]:
def get_schema_linking(db_path, sql):

    schema_dict = get_schema_dict(db_path)
    try:
      info, info_correct = get_tables_and_columns(db_path, sql, schema_dict, True)
      schema_linking = format_dict(info_correct)
    except:
      schema_linking = schema_dict

    return schema_linking

In [ ]:
db_path = '/content/drive/MyDrive/DataCNPJ/cnpjEN.db'
query = """
SELECT establishment.name,
       city.name
FROM establishment
JOIN city ON establishment.city_code = city.code
"""

print(get_schema_linking(db_path, query))

{
  'establishment': ['basic_cnpj', 'name', 'city_code'],
  'city': ['code', 'name']
}
